In [ ]:
import csv
import os
import shutil
import sentencepiece as spm
from google.colab import drive

drive.mount("/content/drive")

drive_dir = "/content/drive/MyDrive/GenAI-Dataset"
train_path = os.path.join(drive_dir, "train.tsv")

ANS_OPEN = "<ans>"
ANS_CLOSE = "</ans>"

train_pairs = []
with open(train_path, "r", encoding="utf-8") as f:
  reader = csv.reader(
      f, delimiter="\t", quoting=csv.QUOTE_NONE, escapechar="\\"
  )
  for row in reader:
    if len(row) == 2:
      train_pairs.append((row[0], row[1]))

print(f"Successfully loaded {len(train_pairs)} pairs from Drive: {train_path}")

corpus_file = "sp_corpus.txt"
with open(corpus_file, "w", encoding="utf-8") as f:
  for src, tgt in train_pairs:
    f.write(src + "\n" + tgt + "\n")

print(f"Corpus prepared: {corpus_file}")

print("Training SentencePiece Tokenizer...")
spm.SentencePieceTrainer.train(
    input=corpus_file,
    model_prefix="ur_sp",
    vocab_size=8000,
    model_type="unigram",
    character_coverage=1.0,
    user_defined_symbols=[ANS_OPEN, ANS_CLOSE],  
    pad_id=0,  # Reserve <pad>
    unk_id=1,  # Reserve <unk>
    bos_id=2,  # Reserve <s>
    eos_id=3,  # Reserve </s>
)

print("Training complete! Files created: ur_sp.model, ur_sp.vocab")

sp = spm.SentencePieceProcessor(model_file="ur_sp.model")
PAD, UNK, BOS, EOS = 0, 1, 2, 3


for i in range(5):
  src, tgt = train_pairs[i]
  tgt_pieces = sp.encode(tgt, out_type=str)
  tgt_ids = sp.encode(tgt)
  round_trip_success = sp.decode(tgt_ids) == tgt

  print(f"\n--- Example {i+1} ---")
  print("Target Question:  ", tgt)
  print("Subword Pieces:   ", tgt_pieces)
  print("Token IDs:        ", tgt_ids)
  print("Round-trip Match: ", round_trip_success)


print("\nSaving generated artifacts to Google Drive...")
shutil.copy("sp_corpus.txt", os.path.join(drive_dir, "sp_corpus.txt"))
shutil.copy("ur_sp.model", os.path.join(drive_dir, "ur_sp.model"))
shutil.copy("ur_sp.vocab", os.path.join(drive_dir, "ur_sp.vocab"))



Mounted at /content/drive
Successfully loaded 75067 pairs from Drive: /content/drive/MyDrive/train.tsv
Corpus prepared: sp_corpus.txt

Training SentencePiece Tokenizer...
Training complete! Files created: ur_sp.model, ur_sp.vocab

TASK 2.3: FIVE TOKENISED EXAMPLES & MORPHOLOGY

--- Example 1 ---
Target Question:   بیونس نے کب مقبولیت حاصل کرنا شروع کی؟
Subword Pieces:    ['▁بیونس', '▁نے', '▁کب', '▁مقبولیت', '▁حاصل', '▁کرنا', '▁شروع', '▁کی', '؟']
Token IDs:         [2757, 18, 83, 2810, 100, 151, 99, 9, 11]
Round-trip Match:  True

--- Example 2 ---
Target Question:   جب وہ بڑی ہو رہی تھی تو بیونس نے کن شعبوں میں مقابلہ کیا؟
Subword Pieces:    ['▁جب', '▁وہ', '▁بڑی', '▁ہو', '▁رہی', '▁تھی', '▁تو', '▁بیونس', '▁نے', '▁کن', '▁شعبوں', '▁میں', '▁مقابلہ', '▁کیا', '؟']
Token IDs:         [91, 70, 196, 224, 621, 49, 165, 2757, 18, 311, 2537, 8, 689, 16, 11]
Round-trip Match:  True

--- Example 3 ---
Target Question:   بیونسی نے ڈسٹنی چائلڈ کب چھوڑ دیا اور سولو گلوکارہ بن گئی؟
Subword Pieces:    ['